In [ ]:
from pathlib import Path
import sys

# Resolve the project from the notebook's working directory (root or any subfolder).
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (start, *start.parents)
     if (folder / "replication_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the replication project or a subfolder of it. "
        "The project root must contain replication_paths.py."
    )
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
from replication_paths import DATASET_DIR, RAW_DEVGPT_DIR, INTERIM_DIR, RESULTS_DIR, HUMAN_FILE, OUTPUT_DIR

from itertools import combinations
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, f1_score

human_file = HUMAN_FILE
llm_dir = RESULTS_DIR
models = ["gpt", "claude", "gemini", "deepseek"]
techniques = ["zero_shot", "few_shot", "mv3"]

In [ ]:
human_to_llm = {
    "Grammar Smell": "grammar_smell",
    "Formatting Smell": "formatting_smell",
    "Readability Smell": "readability_smell",
    "Ambiguous Instruction": "ambiguous_instruction",
    "Missing Constraints": "missing_constraints",
    "Conflicting Instructions": "conflicting_instructions",
    "Intent Misalignment": "intent_misalignment",
    "Missing Context Smell": "missing_context_smell",
    "Irrelevant Context": "irrelevant_context",
    "Complexity Smell": "complexity_smell",
    "Excessive Prompt Length": "excessive_prompt_length",
    "Inefficient Few-shot Examples": "inefficient_few_shot_examples",
    "Overloaded Prompt": "overloaded_prompt",
    "Undefined Output Format": "undefined_output_format",
    "Poor Prompt Organization": "poor_prompt_organization",
    "Role Separation Smell": "role_separation_smell",
    "Incorrect Information": "incorrect_information",
    "Prompt Injection": "prompt_injection",
    "Unsafe Prompt": "unsafe_prompt",
    "Bias Smell": "bias_smell",
    "Style/Formality Mismatch": "style_formality_mismatch",
}
smells = list(human_to_llm.values())
assert len(smells) == 21 and len(set(smells)) == 21

In [ ]:
def human_label(value):
    if pd.isna(value) or str(value).strip() == "":
        return pd.NA
    label = str(value).strip().lower()
    if label in {"0", "0.0", "absent"}: return "0"
    if label in {"1", "1.0", "present"}: return "1"
    if label in {"?", "cannot_determine"}: return "?"
    raise ValueError(f"Unexpected human label: {value!r}")

humans = {}
for reviewer, sheet in [("R1", "Smell_Matrix_R1"), ("R2", "Smell_Matrix_R2")]:
    data = pd.read_excel(human_file, sheet_name=sheet)
    missing = set(human_to_llm) - set(data.columns)
    if missing: raise ValueError(f"Missing columns in {sheet}: {missing}")
    data = data[data["Review Status"].astype(str).str.strip().eq("Reviewed")].copy()
    data["prompt_id"] = data["prompt_id"].astype(str).str.strip()
    if data["prompt_id"].duplicated().any(): raise ValueError(f"Duplicate IDs in {sheet}")
    data = data.set_index("prompt_id")[list(human_to_llm)].rename(columns=human_to_llm)
    humans[reviewer] = data.apply(lambda col: col.map(human_label))

common_human_ids = humans["R1"].index.intersection(humans["R2"].index)
print(f"Human reviewers: {len(common_human_ids)} common prompts, {len(smells)} smells")

In [ ]:
def llm_label(item):
    if item is None: return "0"  # Unlisted smell = not detected
    label = item.get("label") if isinstance(item, dict) else item
    if not isinstance(label, str): return pd.NA
    return {"present": "1", "absent": "0", "cannot_determine": "?"}.get(
        label.strip().lower(), pd.NA)

records = []
for model in models:
    for technique in techniques:
        filename = "classifications.jsonl" if technique == "mv3" else "responses.jsonl"
        path = llm_dir / f"{model}_{technique}" / filename
        with path.open(encoding="utf-8") as file:
            for line in file:
                if not line.strip(): continue
                raw = json.loads(line)
                result = raw.get("result") or raw
                prompt_id = result.get("prompt_id") or raw.get("prompt_id")
                annotations = result.get("smells")
                if not prompt_id or not isinstance(annotations, dict):
                    raise ValueError(f"Invalid result in {path}")
                row = {"model": model, "technique": technique, "prompt_id": str(prompt_id).strip()}
                row.update({smell: llm_label(annotations.get(smell)) for smell in smells})
                records.append(row)

llms = pd.DataFrame(records)
if llms.duplicated(["model", "technique", "prompt_id"]).any():
    raise ValueError("Repeated prompt IDs within an LLM configuration")

In [ ]:
matched = {}
for reviewer, human in humans.items():
    for (model, technique), block in llms.groupby(["model", "technique"]):
        llm = block.set_index("prompt_id")[smells]
        ids = human.index.intersection(llm.index)
        if len(ids) == 0: raise ValueError(f"No matching IDs: {reviewer}, {model}, {technique}")
        matched[(reviewer, model, technique)] = (human.loc[ids], llm.loc[ids])

coverage = pd.DataFrame([
    {"reviewer": r, "model": m, "technique": t, "prompts": len(h)}
    for (r, m, t), (h, _) in matched.items()
])
display(coverage[coverage.technique == "mv3"].reset_index(drop=True))

In [ ]:
agreement_rows = []
for (reviewer, model, technique), (human, llm) in matched.items():
    paired = pd.DataFrame({
        "human": human.to_numpy().ravel(),
        "llm": llm.to_numpy().ravel(),
    }).dropna()
    valid = paired[paired.isin(["0", "1"]).all(axis=1)]
    kappa = cohen_kappa_score(paired.human, paired.llm, labels=["0", "1", "?"])
    agreement_rows.append({
        "reviewer": reviewer, "model": model, "technique": technique,
        "n_labels": len(paired), "agreement": paired.human.eq(paired.llm).mean(),
        "kappa": kappa, "n_binary": len(valid),
        "precision": precision_score(valid.human, valid.llm, pos_label="1", zero_division=0),
        "recall": recall_score(valid.human, valid.llm, pos_label="1", zero_division=0),
        "f1_present": f1_score(valid.human, valid.llm, pos_label="1", zero_division=0),
    })

agreement = pd.DataFrame(agreement_rows)
display(agreement[agreement.technique == "mv3"].round(3).reset_index(drop=True))

In [ ]:
prevalence_rows = []
for (reviewer, model, technique), (human, llm) in matched.items():
    for smell in smells:
        pair = pd.DataFrame({
            "human": human[smell].map({"0": 0, "1": 1}),
            "llm": llm[smell].map({"0": 0, "1": 1}),
        }).dropna()
        if pair.empty: continue
        hp, lp = pair.human.mean(), pair.llm.mean()
        prevalence_rows.append({
            "reviewer": reviewer, "model": model, "technique": technique,
            "smell": smell, "n_valid": len(pair),
            "human_prev": hp, "llm_prev": lp, "diff_pp": 100 * (lp - hp),
        })

prevalence = pd.DataFrame(prevalence_rows)

In [ ]:
rq1_rows = []
for (reviewer, model, technique), group in prevalence.groupby(["reviewer", "model", "technique"]):
    top_h = set(group.nlargest(6, "human_prev").smell)
    top_l = set(group.nlargest(6, "llm_prev").smell)
    rq1_rows.append({
        "reviewer": reviewer, "model": model, "technique": technique,
        "mae_pp": group.diff_pp.abs().mean(),
        "spearman": group.human_prev.corr(group.llm_prev, method="spearman"),
        "top6_overlap": len(top_h & top_l),
    })

rq1_summary = pd.DataFrame(rq1_rows)
display(rq1_summary[rq1_summary.technique == "mv3"].round(3).reset_index(drop=True))

In [ ]:

def pair_metrics(data, a, b):
    n = len(data)
    if n == 0:
        return {"n_ab": 0, "support": np.nan, "conf_a_b": np.nan,
                "conf_b_a": np.nan, "lift": np.nan}
    na, nb = data[a].sum(), data[b].sum()
    nab = ((data[a] == 1) & (data[b] == 1)).sum()
    return {
        "n_ab": int(nab), "support": nab / n,
        "conf_a_b": nab / na if na else np.nan,
        "conf_b_a": nab / nb if nb else np.nan,
        "lift": nab * n / (na * nb) if na and nb else np.nan,
    }

# Preserve the original complete-case pairwise calculations while avoiding
# repeated pandas DataFrame construction for thousands of smell pairs.
cooccurrence_rows = []
for (reviewer, model, technique), (human, llm) in matched.items():
    h_arrays = {smell: pd.to_numeric(human[smell], errors="coerce").to_numpy(dtype=float)
                for smell in smells}
    l_arrays = {smell: pd.to_numeric(llm[smell], errors="coerce").to_numpy(dtype=float)
                for smell in smells}
    for a, b in combinations(smells, 2):
        ha, hb = h_arrays[a], h_arrays[b]
        la, lb = l_arrays[a], l_arrays[b]
        valid = np.isfinite(ha) & np.isfinite(hb) & np.isfinite(la) & np.isfinite(lb)
        both_h = pd.DataFrame({a: ha[valid], b: hb[valid]})
        both_l = pd.DataFrame({a: la[valid], b: lb[valid]})
        h = pair_metrics(both_h, a, b)
        l = pair_metrics(both_l, a, b)
        cooccurrence_rows.append({
            "reviewer": reviewer, "model": model, "technique": technique,
            "smell_a": a, "smell_b": b, "n_valid": int(valid.sum()),
            **{f"human_{key}": val for key, val in h.items()},
            **{f"llm_{key}": val for key, val in l.items()},
        })

cooccurrence = pd.DataFrame(cooccurrence_rows)
cooccurrence["support_diff_pp"] = 100 * (cooccurrence.llm_support - cooccurrence.human_support)


In [ ]:
rq2_rows = []
for (reviewer, model, technique), group in cooccurrence.groupby(["reviewer", "model", "technique"]):
    supports = group[["human_support", "llm_support"]].dropna()
    lifts = group[["human_lift", "llm_lift"]].dropna()
    rq2_rows.append({
        "reviewer": reviewer, "model": model, "technique": technique,
        "support_mae_pp": group.support_diff_pp.abs().mean(),
        "support_spearman": supports.human_support.corr(supports.llm_support, method="spearman"),
        "pairs_with_lift": len(lifts),
        "same_lift_direction": ((lifts.human_lift > 1) == (lifts.llm_lift > 1)).mean()
        if len(lifts) else np.nan,
    })

rq2_summary = pd.DataFrame(rq2_rows)
display(rq2_summary[rq2_summary.technique == "mv3"].round(3).reset_index(drop=True))

In [ ]:
reviewer = "R1"       # Change to "R2" to inspect the second human reviewer
technique = "mv3"    # Or "zero_shot" / "few_shot"

selected = cooccurrence[
    (cooccurrence.reviewer == reviewer) & (cooccurrence.technique == technique)
]
top_pairs = (selected.groupby(["smell_a", "smell_b"])["human_support"]
             .mean().nlargest(10).index)

for metric in ["support", "lift"]:
    table = selected.pivot(index=["smell_a", "smell_b"], columns="model",
                           values=f"llm_{metric}").reindex(top_pairs)
    table.insert(0, "human", selected.groupby(["smell_a", "smell_b"])[f"human_{metric}"]
                 .mean().reindex(top_pairs))
    print(f"{reviewer} versus LLMs — {metric} ({technique})")
    display(table.round(3))

In [ ]:
common_ids = set(humans["R1"].index) & set(humans["R2"].index)

for _, group in llms.groupby(["model", "technique"]):
    common_ids &= set(group["prompt_id"])

ids = sorted(common_ids)
assert ids, "No common prompts found."

def get_prevalence(data):
    binary = data[smells].apply(pd.to_numeric, errors="coerce")
    return binary.mean() * 100

human_r1 = humans["R1"].loc[ids]
human_r2 = humans["R2"].loc[ids]

llm_data = llms[llms["prompt_id"].isin(ids)]

prevalence_all = {
    "Human R1": get_prevalence(human_r1),
    "Human R2": get_prevalence(human_r2),
    "Humans combined": get_prevalence(pd.concat([human_r1, human_r2])),
    "LLMs overall": get_prevalence(llm_data),
}

for model in models:
    prevalence_all[model.upper()] = get_prevalence(
        llm_data[llm_data["model"] == model]
    )

# Top 6 smells with prevalence percentages
names = {v: k for k, v in human_to_llm.items()}

top6 = {
    name: values.nlargest(6)
    for name, values in prevalence_all.items()
}

top6_table = pd.DataFrame({
    name: [
        f"{names.get(smell, smell)} ({value:.1f}%)"
        for smell, value in values.items()
    ]
    for name, values in top6.items()
}, index=range(1, 7))

top6_table.index.name = "Rank"

print(f"Common prompts: {len(ids)}")
display(top6_table)

# Compare each LLM's top 6 against the human top 6
human_top = set(top6["Humans combined"].index)

overlap = pd.DataFrame([
    {
        "Model": name,
        "Top-6 overlap": len(human_top & set(top6[name].index)),
        "Matching smells": ", ".join(
            names.get(s, s)
            for s in top6["Humans combined"].index
            if s in set(top6[name].index)
        )
    }
    for name in ["LLMs overall", "GPT", "CLAUDE", "GEMINI", "DEEPSEEK"]
])

display(overlap)

In [ ]:
comparison_dir = OUTPUT_DIR / "human_llm_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
agreement.to_csv(comparison_dir / "agreement.csv", index=False)
rq1_summary.to_csv(comparison_dir / "prevalence_comparison.csv", index=False)
rq2_summary.to_csv(comparison_dir / "cooccurrence_comparison.csv", index=False)
top6_table.to_csv(comparison_dir / "top6_smells.csv")